# Create Dave config

Builds three independent Dave recipe XMLs from `round_info.csv` and the
Kilroy config for `MICROSCOPE`, in acquisition order: cells, hybs, then the
optional focus-lock test recipe.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/prepare_imaging/<variant>/<acquisition>/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.acquisition.dave   import (
    create_dave_config, dave_config_filename, dave_cells_config_filename,
    dave_focustest_config_filename, create_focus_test_dave_config,
)
from MERci.acquisition.kilroy import find_kilroy_config
from MERci.analysis.stage_z   import read_off_file_if_ready, summarize_focus_lock

In [ ]:
SETTINGS_DIR  = SAMPLE_DIR / "settings"
METADATA_DIR  = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"

# SAMPLE_NAME is the TRUE top-level experiment id -- must match what notebooks
# 03/06 used, since the positions files and round_info rows below are named
# with it (see notebook 03's docstring for why this isn't SAMPLE_DIR.name).
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
# POSITIONS_TAG is what every positions_{...}.txt filename this notebook reads/builds
# actually uses -- SAMPLE_NAME alone in the flat layout, or SAMPLE_NAME_IMAGING_DIR in
# a split layout.
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")

In [ ]:
# ── Experiment parameters ──────────────────────────────────────────
MICROSCOPE           = "ST2"   # must match what you set in notebooks 01/06
USE_ADAPTORS         = False    # True = adaptor-based fluidics; False = direct readouts
INCLUDE_FINAL_CLEAVE = True   # True = add a final cleave step after last imaging round
FIRST_HYB_NO_CLEAVE  = False    # True = first hyb (after the cells round) omits the cleave step

# Default round structure: imaging round 1 = cells only; rounds 2..N_HYBS+1 = bits #1..#N.
# The fluidics before the first bits round has no cleave; later fluidics include the cleave.

# ── Read what notebook 03 produced ──────────────────────────────────
round_info_path = METADATA_DIR / "round_info.csv"
if not round_info_path.exists():
    raise FileNotFoundError(f"{round_info_path} not found -- run notebook 06 first.")
round_info = pd.read_csv(round_info_path)

rbc_path = METADATA_DIR / "round_bit_color_map.csv"
if not rbc_path.exists():
    raise FileNotFoundError(f"{rbc_path} not found -- run notebook 06 first.")
N_HYBS = int(pd.read_csv(rbc_path)["round"].max())

MULTI_BOUNDARY = "positions_file" in round_info.columns
print(f"N_HYBS               : {N_HYBS}")
print(f"Multi-boundary layout: {MULTI_BOUNDARY}")
print(f"Use adaptors         : {USE_ADAPTORS}")
print(f"Final cleave         : {INCLUDE_FINAL_CLEAVE}")
print(f"First hyb no cleave  : {FIRST_HYB_NO_CLEAVE}")

In [ ]:
# ── Resolve positions inputs for the recipe ──────────────────────────────
if MULTI_BOUNDARY:
    # Per-segment: each round_info row names its own positions file in POSITIONS_DIR.
    positions_arg     = None
    positions_dir_arg = POSITIONS_DIR
    missing = [f for f in round_info["positions_file"].unique()
               if not (POSITIONS_DIR / f).exists()]
    if missing:
        raise FileNotFoundError(
            f"Positions files referenced by round_info are missing (run notebook 03): {missing}"
        )
else:
    # Single-positions: one file for every movie.
    positions_arg     = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt"
    positions_dir_arg = None
    if not positions_arg.exists():
        raise FileNotFoundError(f"Positions file not found: {positions_arg}")

# Resolve the Kilroy config that will run this experiment. Its protocol names are
# the source of truth for the fluidic steps written into the Dave recipe, so every
# protocol referenced is guaranteed to exist in Kilroy. If the microscope has no
# Kilroy config, fall back to MF2's.
KILROY_DIR    = MERCI_DIR / "data" / "configs" / "kilroy"
KILROY_CONFIG = find_kilroy_config(MICROSCOPE, KILROY_DIR, fallback_microscope="MF2")
print(f"Kilroy config (protocol source): {KILROY_CONFIG.name}")

# Cells is always imaging_round 1; every later round is a hyb/bits round --
# see this notebook's own "Experiment parameters" cell above.
cells_round_info = round_info[round_info["imaging_round"] == 1]
hybs_round_info  = round_info[round_info["imaging_round"] > 1]

## Cells

Writes the cells-only Dave recipe: a single imaging round, no fluidics at
all (the fluidics that hybridizes bit 1's probes belongs to the hybs recipe
below instead, as its own leading step). Independently runnable in Dave on
its own.

In [ ]:
# ── Build the cells recipe ────────────────────────────────────────────────
cells_output = SETTINGS_DIR / dave_cells_config_filename(MICROSCOPE, SAMPLE_NAME)

create_dave_config(
    round_info            = cells_round_info,
    positions_file        = positions_arg,
    settings_dir          = SETTINGS_DIR,
    output_path           = cells_output,
    kilroy_config         = KILROY_CONFIG,
    positions_dir         = positions_dir_arg,
    include_final_cleave  = False,   # no fluidics belongs in the cells-only file
    microscope            = MICROSCOPE,
)

print(f"Cells Dave config saved: {cells_output}")
with open(cells_output, encoding="ISO-8859-1") as fh:
    print(fh.read())

## Hybs

Writes the hybs-only Dave recipe: every bits/hyb round plus its between-round
fluidics, PLUS its own leading fluidics block before hyb 1
(`leading_fluidics=True`) -- so this file is self-contained and independently
runnable in Dave without depending on the cells recipe above having just
finished in the same session.

In [ ]:
# ── Build the hybs recipe ─────────────────────────────────────────────────
dave_output = SETTINGS_DIR / dave_config_filename(MICROSCOPE, N_HYBS, SAMPLE_NAME)

create_dave_config(
    round_info            = hybs_round_info,
    positions_file        = positions_arg,
    settings_dir          = SETTINGS_DIR,
    output_path           = dave_output,
    use_adaptors          = USE_ADAPTORS,
    include_final_cleave  = INCLUDE_FINAL_CLEAVE,
    first_hyb_no_cleave   = FIRST_HYB_NO_CLEAVE,
    leading_fluidics      = True,
    kilroy_config         = KILROY_CONFIG,
    positions_dir         = positions_dir_arg,
    microscope            = MICROSCOPE,
)

print(f"Hybs Dave config saved: {dave_output}")
with open(dave_output, encoding="ISO-8859-1") as fh:
    print(fh.read())

## Focus-lock test recipe (optional)

Builds a **separate** Dave recipe that visits every FOV and checks focus
lock only -- no fluidics -- to catch a bad lock across the whole coverslip
before committing to the full multi-hour acquisition above. Does not touch
`dave_output`; writes its own file.

**`N_TEST_FRAMES = 0` (default): check-focus only, no image ever taken.**
Confirmed directly against the real, unmodified Dave source (`v2Generator`/
`daveActions`, run against a generated recipe of each kind): a `<movie>`
that omits `<length>`/`<parameters>` expands to a branch containing ONLY
`DAMoveStage` + `DACheckFocus` -- no `DASetParameters`/`DATakeMovie`, no
image taken, no HAL settings changed. No patch to Dave/HAL is needed.

Trade-off: this mode leaves **no persisted per-FOV record** of whether the
lock was good. HAL's focus-status reply only reaches Dave live over TCP;
Dave shows a failed check in its own transient, in-memory warnings list
(never written to a file) -- watch that panel while the recipe runs.

**`N_TEST_FRAMES > 0`: also takes a real (short) movie per FOV.** Needs a
real HAL config/shutter pair -- generated by notebook 01's "create hal
config for focus test" section, alongside the other rounds' HAL configs, so
this notebook just resolves that file by name rather than building it
itself. This produces a genuine, persisted per-FOV record: HAL's normal
`.off` sidecar file (the same one `analysis/stage_z.py` already reads for
stage-z drift), whose `good-offset` column flags exactly which frame(s) had
a bad lock -- at the cost of real (small) disk usage/time per FOV. Read the
results back with the last cell below, after running the recipe on the
microscope.

In [ ]:
# ── Focus-lock test parameters ──────────────────────────────────────────
N_TEST_FRAMES    = 0        # 0 = check-focus only (no movie, no persisted per-FOV
                              # record -- see markdown above); >0 = also take this many
                              # real frames per FOV, producing a real .off file per FOV.
NUM_FOCUS_CHECKS = 50
FOCUS_SCAN       = True

# Focus-test HAL config generated by notebook 01's "create hal config for
# focus test" section -- just resolve its filename here.
_focustest_hal_configs = sorted(SETTINGS_DIR.glob("hal-config-*-focustest-*.xml"))
if not _focustest_hal_configs:
    raise FileNotFoundError(
        f"No focus-test HAL config found in {SETTINGS_DIR} -- run notebook 01's "
        f"'create hal config for focus test' section first."
    )
TEST_HAL_CONFIG = _focustest_hal_configs[-1].name
print(f"Focus-test HAL config: {TEST_HAL_CONFIG}")

# Single-positions layout only for now (notebook 03's all-FOV aggregate file);
# a MULTI_BOUNDARY/multi-tissue layout writes per-tissue files instead, which
# this simple lookup doesn't combine -- set positions_file by hand for that case.
focus_test_positions = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt"
if not focus_test_positions.exists():
    raise FileNotFoundError(f"{focus_test_positions} not found -- run notebook 03 first.")

focus_test_output   = SETTINGS_DIR / dave_focustest_config_filename(MICROSCOPE, SAMPLE_NAME)
focus_test_data_dir = (SAMPLE_DIR / "data" / "focus_test") if N_TEST_FRAMES > 0 else None

n_fovs = create_focus_test_dave_config(
    positions_file   = focus_test_positions,
    output_path      = focus_test_output,
    num_focus_checks = NUM_FOCUS_CHECKS,
    focus_scan       = FOCUS_SCAN,
    n_test_frames    = N_TEST_FRAMES,
    hal_config       = TEST_HAL_CONFIG,
    settings_dir     = SETTINGS_DIR,
    data_dir         = focus_test_data_dir,
    movie_name       = f"hal-{MICROSCOPE.lower()}-focustest",
    kilroy_config    = KILROY_CONFIG,
    microscope       = MICROSCOPE,
)

print(f"Focus-lock test recipe saved: {focus_test_output}")
print(f"FOVs visited: {n_fovs}")
if N_TEST_FRAMES > 0:
    print(f"Each FOV takes a real {N_TEST_FRAMES}-frame movie -> .off files land under {focus_test_data_dir}")
else:
    print("Check-focus only -- no movie taken, no persisted per-FOV record (see markdown above).")

with open(focus_test_output, encoding="ISO-8859-1") as fh:
    print(fh.read())

In [ ]:
# ── Read back per-FOV focus-lock results ──────────────────────────────────
# Only meaningful when N_TEST_FRAMES > 0 above AND the recipe has already been
# run on the microscope (each FOV's .off sidecar only exists once HAL has
# actually written that FOV's movie -- see the markdown above).
if N_TEST_FRAMES > 0:
    # Dave's own auto-increment padding (v2Generator's copyChildren) zero-pads
    # the FOV index to len(str(n_fovs)) digits -- confirmed directly against the
    # real source; distinct from MERci's own fov_pad_width convention used for
    # round_info.csv series patterns, so it is NOT reused here.
    pad = len(str(n_fovs))
    bad_fovs  = []
    n_checked = 0
    for fov_idx in range(n_fovs):
        off_path = focus_test_data_dir / f"hal-{MICROSCOPE.lower()}-focustest_{str(fov_idx).zfill(pad)}.off"
        # read_off_file_if_ready returns None both when the file doesn't exist
        # yet AND when HAL has created it but not finished writing it (a real
        # race when reading back while the recipe is still running on the
        # microscope) -- either way, just means "not ready yet, check again later".
        off_df = read_off_file_if_ready(off_path)
        if off_df is None:
            continue
        n_checked += 1
        summary = summarize_focus_lock(off_df)
        if not summary["all_good"]:
            bad_fovs.append((fov_idx, summary))

    print(f"Checked {n_checked} / {n_fovs} FOV(s) (missing ones haven't been imaged yet).")
    if bad_fovs:
        print(f"{len(bad_fovs)} FOV(s) with a bad focus lock:")
        for fov_idx, summary in bad_fovs:
            print(f"  FOV {fov_idx}: {summary['n_bad_frames']}/{summary['n_frames']} bad frame(s)")
    elif n_checked:
        print("All checked FOVs had a good focus lock.")
    else:
        print("No FOVs found yet -- run the recipe on the microscope first.")
else:
    print("N_TEST_FRAMES == 0 -- no per-FOV file was written (see markdown above); "
          "check Dave's own warnings panel while the recipe runs instead.")